# Legal QA - GPU run on Kaggle: vLLM + AWQ quantization + before/after + load test

> **STATUS.** Real GPU run completed on Kaggle (2x Tesla T4, vLLM 0.31.0, `Qwen/Qwen2.5-1.5B-Instruct` vs its official AWQ checkpoint).
> Results: `reports/optimization.md`, `reports/loadtest/`, `reports/run_info.json`. Dry-run mode (`DRY_RUN = True`) uses a FAKE LLM:
> its numbers describe the stub and must never be reported. Every number from a real run is yours to report; if a step cannot run on
> your GPU, say so rather than estimating.

## What it does
1. Serves a **baseline** LLM and an **AWQ-4bit** version with vLLM (OpenAI-compatible servers)
2. Runs the legal-QA service against the real LLM (`prodml-ask`, `prodml-smoke`)
3. **Compares baseline vs quantized**: latency p50/p95, tokens/s, faithfulness (lexical *proxy*; optional real RAGAS) -> `optimization.md`
4. **Load-tests the service at 50 users** against the real LLM (Locust) -> `loadtest/`
5. Zips everything into `results.zip`

## How to run it on Kaggle (about 10 minutes of setup, 30-45 min of run time)
**A. Put the code where the notebook can find it** (pick one):
- **Easiest - upload the project as a Dataset:** kaggle.com -> *Datasets* -> *New Dataset* -> upload `mlops-legal-qa-v5.tar.gz` (the file I gave you), name it e.g. `legal-qa-code`.
  In this notebook click *Add Input* (right sidebar) -> *Your Datasets* -> select it. No tokens needed.
- **Or clone your private GitHub repo:** *Add-ons -> Secrets* -> add a secret named `GH_TOKEN` (a fine-grained GitHub token with read access to the repo, **never** pasted into a cell) and *attach* it to the notebook.

**B. Import this notebook:** *Create -> New Notebook -> File -> Import Notebook* -> upload `kaggle_gpu_run.ipynb`.

**C. Do a free dry run first** (saves your ~30 h/week GPU quota): *Settings -> Accelerator = None*, *Internet = On*, leave `DRY_RUN = True` below, **Run All**.
It uses a fake LLM, so it only proves the notebook works. Fix anything that fails *here*, not on the GPU clock.

**D. The real run:** *Settings -> Accelerator = **GPU T4 x2*** (**not P100** - vLLM does not support it) and *Internet = On*. Set `DRY_RUN = False`. **Run All**.

**E. Get the results:** *Save Version -> Save & Run All (Commit)* then download from the *Output* tab, or download `results.zip` from the file panel.

## 1. Configuration - the only cell you normally edit

In [2]:
# ===================== CONFIG =====================
DRY_RUN = False              # True: FAKE LLM server, works on a free CPU session, no GPU. False: the real run (needs GPU T4 x2).
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # <-- the model name to document in the README
QUANT_OPTION = "official"   # "official": the publisher's own AWQ checkpoint (reliable)  |  "own": quantize it yourself with AutoAWQ (handbook's wording)
LOAD_USERS = 50             # concurrent users in the load test (handbook: 50)
LOAD_SECONDS = 90           # load-test length (a dry run always uses 15 s)
LLM_TIMEOUT_S = 120         # per-request LLM timeout. 50 users on one T4 queue up: the 30 s default would silently fall back to extractive answers
RUN_RAGAS = False           # real RAGAS faithfulness (needs a judge-LLM key in the secret OPENAI_API_KEY; costs money)
VLLM_VERSION = "0.31.0"     # the version the reported run used; empty = latest
VLLM_EXTRA_ARGS = []        # e.g. ["--enforce-eager"] if the vLLM server fails to start (more compatible, slower)
BASE_PORT, QUANT_PORT, API_PORT = 8001, 8002, 8000
# ==================================================
OUT = "reports/dry_run" if DRY_RUN else "reports"
if DRY_RUN:
    print("=" * 78 + "\nDRY RUN: a FAKE LLM is used. Numbers describe the stub, NOT a model. Output goes to reports/dry_run/\n" + "=" * 78)
else:
    print("REAL RUN: needs a GPU, Internet = On. Output goes to reports/")

REAL RUN: needs a GPU, Internet = On. Output goes to reports/


## 2. Check the hardware
vLLM needs an NVIDIA GPU with compute capability >= 7.0, and AWQ kernels need >= 7.5. **Kaggle's T4 is 7.5 (fine); the P100 is 6.0 (will not work).**

In [3]:
import os, sys, json, glob, shutil, subprocess, time, urllib.request, atexit

def gpu_info():
    if not shutil.which("nvidia-smi"):
        return []
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
    return [[p.strip() for p in ln.split(",")] for ln in r.stdout.strip().splitlines() if ln.strip()]

# Make the commands installed into THIS kernel's environment (prodml-*, uvicorn, locust, vllm) findable by `!` lines and subprocesses.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

GPUS = gpu_info()
N_GPUS = len(GPUS)
print("GPUs:", GPUS or "none")
if not DRY_RUN:
    if N_GPUS == 0:
        raise RuntimeError("No GPU found. On Kaggle: Settings -> Accelerator -> GPU T4 x2 (and Internet = On). Or set DRY_RUN = True.")
    try:
        cap = min(float(g[1]) for g in GPUS)
    except (ValueError, IndexError):
        cap = None
    if cap is not None and cap < 7.5:
        raise RuntimeError(f"GPU compute capability {cap} is too old for vLLM + AWQ (need >= 7.5). Choose 'GPU T4 x2', not P100.")
print("OK" if not DRY_RUN else "dry run: GPU not required")

GPUs: [['Tesla T4', '7.5', '15360 MiB'], ['Tesla T4', '7.5', '15360 MiB']]
OK


## 3. Find the project code
Looks for the project in this order: (1) already in the current folder, (2) a Kaggle input (a dataset folder, `.tar.gz`, or `.zip`), (3) a `git clone` using the `GH_TOKEN` secret.

In [4]:
import tarfile, zipfile

def is_project(d):
    return os.path.exists(os.path.join(d, "pyproject.toml")) and os.path.isdir(os.path.join(d, "src", "prodml"))

def find_in(base):
    for pj in glob.glob(f"{base}/**/pyproject.toml", recursive=True):
        d = os.path.dirname(pj)
        if is_project(d):
            return d
    return None

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
ROOT = os.getcwd() if is_project(os.getcwd()) else None

if ROOT is None:                                   # (2a) an extracted dataset folder
    for base in [b for b in ("/kaggle/input", "/content") if os.path.isdir(b)]:
        ROOT = ROOT or find_in(base)

if ROOT is None:                                   # (2b) an archive in the inputs
    for base in [b for b in ("/kaggle/input", "/content") if os.path.isdir(b)]:
        for pat in ("*.tar.gz", "*.tgz", "*.tar", "*.zip"):
            for arc in glob.glob(f"{base}/**/{pat}", recursive=True):
                dest = os.path.join(WORK, "project")
                os.makedirs(dest, exist_ok=True)
                (zipfile.ZipFile(arc) if arc.endswith(".zip") else tarfile.open(arc)).extractall(dest)
                ROOT = find_in(dest)
                if ROOT:
                    print("extracted", arc)
                    break
            if ROOT: break
        if ROOT: break

if ROOT is None:                                   # (3) git clone with a secret token
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
        except Exception:
            pass
    if token:
        dest = os.path.join(WORK, "repo")
        try:
            subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/badawyengineer/MLOps_community.git", dest],
                           check=True, capture_output=True)
            subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", "https://github.com/badawyengineer/MLOps_community.git"], check=True)
            ROOT = find_in(dest)
        except subprocess.CalledProcessError:
            print("git clone failed (check the token's access to the repo). The token is not printed on purpose.")

if ROOT is None:
    raise RuntimeError("Project not found. Either add the uploaded Dataset via 'Add Input', or add a GH_TOKEN secret (see the top of this notebook).")
if ROOT.startswith("/kaggle/input") or not os.access(ROOT, os.W_OK):
    # Kaggle inputs are READ-ONLY, but pip and the reports need to write into the project: work on a copy.
    copy = os.path.join(WORK, "project_copy")
    shutil.copytree(ROOT, copy, dirs_exist_ok=True)
    print("input is read-only; working on a copy:", copy)
    ROOT = copy
os.chdir(ROOT)
print("project root:", ROOT)

input is read-only; working on a copy: /kaggle/working/project_copy
project root: /kaggle/working/project_copy


## 4. Install
Installs the project, and (real run only) vLLM. **If pip says it replaced PyTorch, restart the session** (*Run -> Restart session*) and run again from the top; that is normal for vLLM.

In [5]:
!{sys.executable} -m pip install -q -e ".[dev,loadtest]"
if not DRY_RUN:
    spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
    !{sys.executable} -m pip install -q "{spec}"
!prodml-index
print("installed")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 90.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2

## 5. Quantization source
- `QUANT_OPTION = "official"` (default): use the publisher's AWQ checkpoint of the same model (`<model>-AWQ`). Reliable. **You must say in the report that you did not quantize it yourself.**
- `QUANT_OPTION = "own"`: quantize it here with AutoAWQ in a **separate virtualenv** (its pinned torch/transformers can clash with vLLM's). Needs a GPU and a few extra minutes. Falls back to the official checkpoint if anything fails.

In [6]:
QUANT_MODEL = MODEL + "-AWQ"
QUANT_SOURCE = "official AWQ checkpoint published by the model authors (NOT quantized by this notebook)"

if DRY_RUN:
    print("dry run: no quantization; the 'quantized' server is a second fake server")
elif QUANT_OPTION == "own":
    !{sys.executable} -m venv awq_env
    !awq_env/bin/pip install -q autoawq transformers
    !awq_env/bin/python scripts/quantize_awq.py --model {MODEL} --out qwen-awq
    if os.path.exists("qwen-awq/config.json"):
        QUANT_MODEL, QUANT_SOURCE = os.path.abspath("qwen-awq"), "quantized by this notebook with AutoAWQ (w_bit=4, group_size=128, GEMM)"
    else:
        print("Own quantization FAILED - falling back to the official AWQ checkpoint. Say so in the report.")
print("quantized model:", QUANT_MODEL, "|", QUANT_SOURCE)

quantized model: Qwen/Qwen2.5-1.5B-Instruct-AWQ | official AWQ checkpoint published by the model authors (NOT quantized by this notebook)


## 6. Start the two LLM servers
Baseline on GPU 0, quantized on GPU 1 when two GPUs exist (otherwise they share one GPU at 40% memory each). `--dtype half` because the T4 has no bfloat16.
The first start downloads the model and can take several minutes.

### Environment fix for vLLM on Kaggle
Needed on the image used for the reported run (Python 3.13, PyTorch 2.11 cu128, driver CUDA 13). Run it before starting the servers.

In [ ]:
# vLLM on this Kaggle image needs the CUDA 13 runtime libraries on LD_LIBRARY_PATH (found while debugging the first real run).
# Inherited by the vllm subprocesses started below. Harmless if the folder does not exist.
import os

cu13 = "/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib"

os.environ["LD_LIBRARY_PATH"] = (
    cu13 + ":" + os.environ.get("LD_LIBRARY_PATH", "")
)

print("CUDA 13 library path:")
print(cu13)
print("\nLD_LIBRARY_PATH:")
print(os.environ["LD_LIBRARY_PATH"])

In [20]:
PROCS = []

def cleanup():
    for p in PROCS:
        if p.poll() is None:
            p.terminate()

atexit.register(cleanup)


def wait_http(url, seconds, proc=None, logfile=None):
    for _ in range(int(seconds / 2)):
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status < 500:
                    return
        except Exception:
            if proc is not None and proc.poll() is not None:
                tail = ""
                if logfile and os.path.exists(logfile):
                    with open(
                        logfile,
                        encoding="utf-8",
                        errors="replace"
                    ) as f:
                        tail = f.read()[-5000:]

                raise RuntimeError(
                    f"\nServer exited early.\n"
                    f"URL: {url}\n"
                    f"Log: {logfile}\n\n"
                    f"Last log lines:\n{tail}"
                )

            time.sleep(2)

    raise TimeoutError(
        f"{url} did not come up in {seconds}s. "
        f"Check {logfile}"
    )


def start_server(model, port, gpu, latency_ms):
    logfile = f"server_{port}.log"

    env = os.environ.copy()

    if DRY_RUN:
        cmd = [
            "prodml-mock-llm",
            "--port", str(port),
            "--model", model,
            "--latency-ms", str(latency_ms),
        ]

    else:
        # Select GPU
        env["CUDA_VISIBLE_DEVICES"] = str(
            gpu if N_GPUS >= 2 else 0
        )

        mem = "0.85" if N_GPUS >= 2 else "0.40"

        cmd = [
            "vllm",
            "serve",
            model,

            "--port", str(port),

            # Better to let vLLM choose the appropriate dtype
            "--dtype", "auto",

            "--max-model-len", "4096",

            "--gpu-memory-utilization", mem,
        ]

        cmd.extend(VLLM_EXTRA_ARGS)

        # AWQ models
        if "-AWQ" in model or model.endswith("qwen-awq"):
            cmd.extend([
                "--quantization",
                "awq"
            ])

    print(f"\nStarting server:")
    print(f"  Model : {model}")
    print(f"  GPU   : {gpu}")
    print(f"  Port  : {port}")
    print(f"  Log   : {logfile}")

    log_file = open(logfile, "w")

    proc = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
    )

    PROCS.append(proc)

    wait_http(
        f"http://localhost:{port}/health",
        900 if not DRY_RUN else 60,
        proc,
        logfile,
    )

    print(f"✓ up: {model} on :{port}")

    return proc



baseline = start_server(
    MODEL,
    BASE_PORT,
    gpu=0,
    latency_ms=60,
)

quant = start_server(
    QUANT_MODEL,
    QUANT_PORT,
    gpu=1,
    latency_ms=45,
)



if not DRY_RUN:
    !nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv


Starting server:
  Model : Qwen/Qwen2.5-1.5B-Instruct
  GPU   : 0
  Port  : 8001
  Log   : server_8001.log
✓ up: Qwen/Qwen2.5-1.5B-Instruct on :8001

Starting server:
  Model : Qwen/Qwen2.5-1.5B-Instruct-AWQ
  GPU   : 1
  Port  : 8002
  Log   : server_8002.log
✓ up: Qwen/Qwen2.5-1.5B-Instruct-AWQ on :8002
index, name, memory.used [MiB], memory.total [MiB]
0, Tesla T4, 12383 MiB, 15360 MiB
1, Tesla T4, 12451 MiB, 15360 MiB


## 7. Run the real service against the LLM

In [24]:
os.environ.update(PRODML_GENERATION_BACKEND="vllm",
                  PRODML_LLM_BASE_URL=f"http://localhost:{BASE_PORT}/v1",
                  PRODML_LLM_MODEL=MODEL,
                  PRODML_LLM_TIMEOUT=str(LLM_TIMEOUT_S))
!prodml-ask "Must contracts be performed in good faith?"
!prodml-ask "هل يجوز تعديل العقد"
!prodml-smoke

Yes, according to Article 148, a contract must be performed in accordance with its contents and in a manner consistent with the requirements of good faith.

Sources: Egyptian Civil Code, Article 148; Egyptian Civil Code, Article 147; Egyptian Civil Code, Article 163
Confidence: 0.6028   Faithfulness proxy: 1.0
يجوز تعديل العقد بناءً على مقتضى العرف أو بموجب الأسباب التي يقررها القانون.

Sources: Egyptian Civil Code, Article 147; Egyptian Civil Code, Article 148; Egyptian Civil Code, Article 1
Confidence: 0.3447   Faithfulness proxy: 0.0
Smoke test (in-process - no server needed)

  PASS  GET  /health                           5 documents indexed
  PASS  GET  /metadata                         retriever=hybrid-dense generation=vllm
  PASS  POST /ask (English)                    cites Egyptian Civil Code, Article 148 (confidence 0.6028)
  PASS  POST /ask (Arabic)                     cites Egyptian Civil Code, Article 147
  PASS  POST /ask (out of scope -> abstain)    refused an out-of-sco

## 8. Before / after: baseline vs quantized
Retrieval is shared, so any difference comes from the generator alone. Faithfulness here is the lexical **proxy**; the handbook's *RAGAS* criterion (drop <= 0.03) needs section 10.

In [25]:
!mkdir -p {OUT}
!prodml-compare-llms --a-url http://localhost:{BASE_PORT}/v1 --a-model {MODEL} --b-url http://localhost:{QUANT_PORT}/v1 --b-model {QUANT_MODEL} --out {OUT}/optimization.md --answers-json {OUT}/answers.json
!cat {OUT}/optimization.md

| metric | A: Qwen/Qwen2.5-1.5B-Instruct | B: Qwen/Qwen2.5-1.5B-Instruct-AWQ |
|---|---|---|
| latency p50 (ms) | 628.10 | 181.08 |
| latency p95 (ms) | 2121.24 | 3101.26 |
| tokens / s | 60.97 | 98.61 |
| faithfulness (proxy) | 0.52 | 0.38 |

Faithfulness-proxy drop A->B: +0.143 (EXCEEDS the 0.03 budget). n=16/16 questions.
| metric | A: Qwen/Qwen2.5-1.5B-Instruct | B: Qwen/Qwen2.5-1.5B-Instruct-AWQ |
|---|---|---|
| latency p50 (ms) | 628.10 | 181.08 |
| latency p95 (ms) | 2121.24 | 3101.26 |
| tokens / s | 60.97 | 98.61 |
| faithfulness (proxy) | 0.52 | 0.38 |

Faithfulness-proxy drop A->B: +0.143 (EXCEEDS the 0.03 budget). n=16/16 questions.


## 9. Load test: 50 concurrent users, real LLM in the loop
Starts the API against the baseline LLM and hammers it with Locust. (The repository's earlier load test used the extractive generator, so the LLM was not in the loop.)

In [26]:
import csv

def summarize_llm_metrics(text):
    """What the LLM really did during the load test, read from the API's own /metrics (Prometheus text format)."""
    import re
    series = {}
    for line in text.splitlines():
        m = re.match(r"^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{[^}]*\})?\s+([-+0-9.eE]+|NaN)$", line)
        if m and m.group(3) != "NaN":
            series.setdefault(m.group(1), []).append((m.group(2) or "", float(m.group(3))))
    def total(name, contains=""):
        return sum(v for lbl, v in series.get(name, []) if contains in lbl)
    n = total("prodml_faithfulness_proxy_count")
    return {
        "answers": total("prodml_requests_total", 'path="/ask"') + total("prodml_requests_total", 'path="/ask/stream"'),
        "llm_fallbacks": total("prodml_llm_fallbacks_total"),
        "prompt_tokens": total("prodml_llm_tokens_total", 'kind="prompt"'),
        "completion_tokens": total("prodml_llm_tokens_total", 'kind="completion"'),
        "abstentions": total("prodml_abstentions_total"),
        "unsupported_citations": total("prodml_guardrail_flags_total", 'flag="unsupported_citation"'),
        "faithfulness_proxy_mean": (total("prodml_faithfulness_proxy_sum") / n) if n else None,
    }

!mkdir -p {OUT}/loadtest
api = subprocess.Popen(["uvicorn", "prodml.api.main:app", "--port", str(API_PORT), "--log-level", "warning"],
                       stdout=open("api.log", "w"), stderr=subprocess.STDOUT)
PROCS.append(api)
wait_http(f"http://localhost:{API_PORT}/health", 120, api, "api.log")
secs = 15 if DRY_RUN else LOAD_SECONDS
!locust -f loadtest/locustfile.py --host http://localhost:{API_PORT} --headless -u {LOAD_USERS} -r 5 -t {secs}s --csv {OUT}/loadtest/ask --html {OUT}/loadtest/ask.html
LLM_SUMMARY = summarize_llm_metrics(urllib.request.urlopen(f"http://localhost:{API_PORT}/metrics", timeout=10).read().decode())
api.terminate()

print(f"\n{'endpoint':<16}{'requests':>9}{'failures':>9}{'req/s':>7}{'p50 ms':>8}{'p95 ms':>8}{'p99 ms':>8}")
for r in csv.DictReader(open(f"{OUT}/loadtest/ask_stats.csv", encoding="utf-8")):
    print(f"{r['Name']:<16}{r['Request Count']:>9}{r['Failure Count']:>9}{float(r['Requests/s']):>7.1f}{r['50%']:>8}{r['95%']:>8}{r['99%']:>8}")
exc = open(f"{OUT}/loadtest/ask_exceptions.csv", encoding="utf-8").read().strip().splitlines()
print("\nlocustfile exceptions:", "none" if len(exc) <= 1 else f"{len(exc) - 1} - READ {OUT}/loadtest/ask_exceptions.csv before trusting these numbers")

print("\nWhat the LLM actually did during the load test (from the API's own metrics):")
for k, v in LLM_SUMMARY.items():
    print(f"  {k:<26}", v if v is None else round(v, 3))
if LLM_SUMMARY["llm_fallbacks"]:
    print(f"\n!! {int(LLM_SUMMARY['llm_fallbacks'])} of {int(LLM_SUMMARY['answers'])} answers FELL BACK to extractive text because the LLM call failed or timed out.")
    print("   The latency and throughput above are therefore NOT purely the LLM. Raise LLM_TIMEOUT_S or lower LOAD_USERS and re-run BEFORE reporting.")
elif not DRY_RUN and not LLM_SUMMARY["completion_tokens"]:
    print("\n!! No LLM tokens were generated: the LLM was not in the loop. Do NOT report these numbers.")
else:
    print("\nOK: no fallbacks - every answer in this test came from the LLM" + (" (a FAKE one: dry run)." if DRY_RUN else "."))

[2026-10-08 17:15:13,695] 2d299b762e58/INFO/locust.main: Starting Locust 2.46.7
[2026-10-08 17:15:13,696] 2d299b762e58/INFO/locust.main: Run time limit set to 90 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-10-08 17:15:13,698] 2d299b762e58/INFO/locust.runners: Ramping to 50 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /ask [abstain]       2     0(0.00%) |     15       5      24     24 |    0.00        0.00
POST     /ask [ar]       2     0(0.00%) |    569     546     592    590 |    0.00        0.00

## 10. (Optional) Real RAGAS faithfulness before vs after
Set `RUN_RAGAS = True`, add your judge-LLM key as a Kaggle secret named `OPENAI_API_KEY`, and attach it. **This costs real API money.** It replaces the lexical proxy with the number the handbook asks for (drop <= 0.03).

In [27]:
if RUN_RAGAS and not DRY_RUN:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
    !{sys.executable} -m pip install -q -e ".[ragas]"
    from prodml.rag import LegalQAPredictor
    from prodml.faithfulness import ragas_scores
    p = LegalQAPredictor().load()
    answers = json.load(open(f"{OUT}/answers.json", encoding="utf-8"))
    def rows_for(side):
        rows = []
        for a in answers[side]:
            r = p._retrieve(a["question"])
            ctx = p.contexts_for([m.article["citation"] for m in r.matches])
            rows.append({"question": a["question"], "answer": a["answer"], "contexts": ctx, "ground_truth": ctx[0] if ctx else ""})
        return rows
    A, B = ragas_scores(rows_for("A")), ragas_scores(rows_for("B"))
    print("baseline :", A); print("quantized:", B)
    print("RAGAS faithfulness drop:", round(A["faithfulness"] - B["faithfulness"], 3), "(budget 0.03)")
    json.dump({"baseline": A, "quantized": B}, open(f"{OUT}/ragas_before_after.json", "w"), indent=2)
else:
    print("skipped (RUN_RAGAS is False or this is a dry run)")

skipped (RUN_RAGAS is False or this is a dry run)


## 11. Package the results and shut everything down

In [28]:
import importlib.metadata as md_
try:
    vllm_version = md_.version("vllm")
except Exception:
    vllm_version = None
os.makedirs(f"{OUT}/logs", exist_ok=True)
for f in glob.glob("server_*.log") + glob.glob("api.log"):
    shutil.copy(f, f"{OUT}/logs/")
json.dump({"dry_run": DRY_RUN, "model": MODEL, "quantized_model": QUANT_MODEL, "quantization": QUANT_SOURCE,
           "gpus": GPUS, "vllm_version": vllm_version, "load_users": LOAD_USERS, "llm_timeout_s": LLM_TIMEOUT_S, "load_test_llm_summary": LLM_SUMMARY,
           "note": "DRY RUN numbers describe a fake stub and are not results." if DRY_RUN else "real run"},
          open(f"{OUT}/run_info.json", "w"), indent=2)
zip_base = os.path.join(WORK, "results_dry_run" if DRY_RUN else "results")
print("created", shutil.make_archive(zip_base, "zip", ROOT, OUT))
for p_ in PROCS:
    if p_.poll() is None:
        p_.terminate()
print("servers stopped")

created /kaggle/working/results.zip
servers stopped


## What to do with the results (real run only)
1. Copy `reports/` into your repo; commit and push.
2. In the README, write: the **model name**, whether the quantization was **yours or the publisher's** (see `run_info.json`), and the **measured** latency and faithfulness numbers.
3. If you ran section 10, the RAGAS drop is the handbook's number; otherwise report the proxy **as a proxy**.
4. Tick the checklist items in `reports/final-checklist.md` only for steps that actually ran.

## Troubleshooting
| Symptom | Likely cause / fix |
|---|---|
| `No GPU found` / `compute capability ... too old` | Settings -> Accelerator -> **GPU T4 x2** (not P100); Internet = On |
| `Project not found` | Use *Add Input* to attach your uploaded Dataset, or attach a `GH_TOKEN` secret |
| vLLM server exits early (log is printed) | Try `VLLM_EXTRA_ARGS = ["--enforce-eager"]`, or pin `VLLM_VERSION` to an older release; check the log for out-of-memory |
| pip replaced PyTorch / import errors after section 4 | *Run -> Restart session*, then Run All again |
| `Own quantization FAILED` | AutoAWQ is no longer actively maintained; keep the fallback and state in the report that the AWQ weights are the publisher's |
| Model download is slow or fails | Internet must be On; try again, or use a smaller model |
| Quota running out | Use the dry run to debug; real runs need ~30-45 min |
